In [94]:
## scheme: build the data-->obtain R and C-->compute the risck function-->Simulated Annealing-->Clean Covariance Matrix


In [99]:
## Data:
import numpy as np

import pandas as pd

import yfinance as yf

import matplotlib.pyplot as plt

tickers = [

    "AAPL", "MSFT", "AMZN", "GOOGL", "META",

    "NVDA", "TSLA", "JPM", "V", "MA",

    "UNH", "JNJ", "PG", "HD", "XOM",

    "CVX", "KO", "PEP", "MRK", "ABBV",

    "PFE", "BAC", "WFC", "GS", "MS",

    "DIS", "NFLX", "ADBE", "CRM", "ORCL",

    "INTC", "AMD", "CSCO", "QCOM", "TXN",

    "MCD", "NKE", "WMT", "COST", "TGT",

    "BA", "CAT", "GE", "HON", "UPS",

    "LOW", "IBM", "AMGN", "TMO", "LIN"

]

start_date = "2018-01-01"

end_date = "2024-12-31"

data = yf.download(

    tickers,

    start=start_date,

    end=end_date,

    auto_adjust=True

)

prices = data["Close"]

prices = prices.dropna()

print(prices.shape)
prices.tail()


[*********************100%***********************]  50 of 50 completed

(1760, 50)


Ticker,AAPL,ABBV,ADBE,AMD,AMGN,AMZN,BA,BAC,CAT,COST,...,TGT,TMO,TSLA,TXN,UNH,UPS,V,WFC,WMT,XOM
Date,,,,,,,,,,,,,,,,,,,,,
2024-12-23,253.649429,169.580612,446.739990,124.599998,252.015106,225.059998,177.690002,42.434864,358.431732,942.322388,...,123.903275,522.921936,430.600006,182.041641,490.358002,114.120636,313.684296,68.311554,89.139961,101.171150
2024-12-24,256.560822,171.111115,447.940002,126.290001,252.482819,229.050003,179.339996,42.908615,360.570221,951.161194,...,124.354095,526.595276,462.279999,184.243668,490.125610,114.583481,317.076080,69.328369,91.438759,101.266327
2024-12-26,257.375580,170.350632,450.160004,125.059998,251.232315,227.050003,180.380005,43.072983,360.128754,948.502747,...,128.101624,524.803406,454.130005,183.563919,495.016205,114.674217,317.333221,69.493011,91.547287,101.351974
2024-12-27,253.967392,169.219376,446.480011,125.190002,250.726364,223.750000,180.720001,42.869942,357.911774,932.194031,...,127.265701,523.698425,431.660004,183.037323,493.892822,114.447350,315.108246,68.863541,90.432419,101.342468
2024-12-30,250.598923,167.498764,445.799988,122.440002,247.528442,221.300003,176.550003,42.454201,356.097046,914.843750,...,126.843079,516.501099,417.410004,179.954498,491.771942,113.730408,311.795593,68.185654,89.357010,100.657204


In [100]:

returns= np.log(prices/prices.shift(1))
returns=returns.dropna()
returns.tail()
#print(returns.shape)
##1760 rapprensentano le righe, ovvero il numero di giorni osservati, 50 sono i titoli che stiamo studiando
##non utilizziamo tutti i dati, li dividiamo in training e test. Quindi devo ottenere due tabelle.
#print(len(returns))

Ticker,AAPL,ABBV,ADBE,AMD,AMGN,AMZN,BA,BAC,CAT,COST,...,TGT,TMO,TSLA,TXN,UNH,UPS,V,WFC,WMT,XOM
Date,,,,,,,,,,,,,,,,,,,,,
2024-12-23,0.003060,0.015877,-0.000962,0.044222,0.002351,0.000622,0.001915,-0.006359,-0.001777,-0.004370,...,0.003341,0.002363,0.022404,0.017348,0.012340,0.000557,-0.001544,0.002839,-0.020703,0.004053
2024-12-24,0.011413,0.008985,0.002683,0.013472,0.001854,0.017573,0.009243,0.011102,0.005949,0.009336,...,0.003632,0.007000,0.070991,0.012024,-0.000474,0.004048,0.010755,0.014775,0.025462,0.000940
2024-12-26,0.003171,-0.004454,0.004944,-0.009787,-0.004965,-0.008770,0.005782,0.003823,-0.001225,-0.002799,...,0.029691,-0.003409,-0.017787,-0.003696,0.009929,0.000792,0.000811,0.002372,0.001186,0.000845
2024-12-27,-0.013331,-0.006663,-0.008208,0.001039,-0.002016,-0.014641,0.001883,-0.004725,-0.006175,-0.017344,...,-0.006547,-0.002108,-0.050745,-0.002873,-0.002272,-0.001980,-0.007036,-0.009099,-0.012253,-0.000094
2024-12-30,-0.013352,-0.010220,-0.001524,-0.022211,-0.012837,-0.011010,-0.023345,-0.009745,-0.005083,-0.018788,...,-0.003326,-0.013839,-0.033569,-0.016986,-0.004303,-0.006284,-0.010568,-0.009893,-0.011963,-0.006785


In [101]:
##dati di training e test:

index=int(len(returns)*0.7)


returns_train=returns.iloc[:index] 
returns_test=returns.iloc[index:]

print(returns_train.shape)
print(returns_test.shape)
print(len(returns_train)+len(returns_test))


(1231, 50)
(528, 50)
1759


In [130]:
##Media rendimenti e matrice di covarianza:

mean=returns_train.mean().values ##restitusce la media gironaliera dei 50 titoli, nel periodo di training
covariance=returns_train.cov().values
#print(mean.shape)
#print(covariance.shape)



##lavoriamo su base annua:
mean_year=mean*252
cov_year=covariance*252

##ora dobbiamo impostare il nostro problmea:
max_asset=10
x_0=np.array([])##creiamo il portafolgio inziale, con 10 titoli
for j in range(0,3):
    x_0=np.append(x_0,0.1)
for j in range(0,57):
    x_0=np.append(x_0,0)

C = np.array(cov_year, dtype=np.float64)
x0 = np.array(x_0, dtype=np.float64)

##calcolo il rendimento(R_0) del portafolgio inziale e il rischio(F_0):
R_0=mean_year.T@x0
print(f'Questo è il rendimento annuo con il portafoglio inizale:{R_0}')
temp = C.dot(x0)
F0 = x0.dot(temp)
print(f'Questo è la funzione inziale, rispetto al portafolgio inizale:{F0}')
R_exp=R_0
N=10
xmin=0
xmax=0.20

def controllo(portafoglio):
    errore=[]
    n=portafoglio[(portafolgio>0.2) | (portafoglio<0)]
    if len(n)>0:
        errore.append("il limite massimo o minimo non è verificato")
    k=portafoglio[portafolgio>0]
    if len(k)>10:
        errore.append("ci sono piu asset del massimo")
    somma=portafolglio.sum()
    if somma!=1:
        errore.append("il portafoglio non è normalizzato")
    rendimento=mean_year@portafoglio
    if rendimento!=R_exp:
        errore.append("il rendimento non è uguale a quello fissato")

    return errore


##funzione obbiettivo:

def funzione_obiettivo(portafoglio):
    temp = C.dot(portafoglio)
    F = x.dot(portafoglio)
    return F


##generazione di un vicino:




def vicino(portafoglio):
    #il primo asset va scelto in modo particolare:
    indici_mediayear_ordinati=np.argsort(mean_year)
    media_ordinata=mean_year[indici_mediayear_ordinati]
    differenza=np.abs(media_ordinata-R_exp)
    q=np.argmin(differenza)##sto cercando l'indice dell'asset piu vicino al rendimento fissato, nella lista ordinata
    ##indice dell'asset nell'array originale

    posizione1=np.random.normal(loc=q,scale=12.5)
    posizione1 = np.clip(posizione1, 0, len(mean_year)-1)#se valore è più piccolo del minimo, mettilo uguale al minimo etc..
    posizione1=int(round(posizione1))##approssima il numero al piu vicino
    indice1_array=np.array([indici_mediayear_ordinati[posizione1]])##indice dell'array originale degli asset
    indice1=indici_mediayear_ordinati[posizione1]
    asset_usati=np.where(portafoglio > 0)[0]##array che contiene gli indici di asset usati
    asset_nuovi=np.where(portafoglio == 0)[0]
    numero_asset_usati=asset_usati.size
    posti_liberi_portafoglio=N-numero_asset_usati
 
    asset_usati=asset_usati[asset_usati!=indice1]##rimuoviamo indice1 al fine di non scelgierlo dopo
    asset_nuovi=asset_nuovi[asset_nuovi!=indice1]
    if posti_liberi_portafoglio>=0:
        if posti_liberi_portafoglio>=2:
            numero_scelte=np.random.randint(0,3)
            indici_usati_scelti = np.random.choice(asset_usati, size=numero_scelte, replace=False)##scelgi casualemtne dei numeri in asse_usati(indici), tutti diversi tra di loro
            
        
            numero_nuovi=2-indici_usati_scelti.size
            indici_nuovi_scelti=np.random.choice(asset_nuovi, size=numero_nuovi, replace=False)
            indici_totali=np.concatenate((indice1_array,indici_usati_scelti))
            indici_totali=np.concatenate((indici_totali, indici_nuovi_scelti))
            

            
        elif posti_liberi_portafoglio==1:
            numero_scelte=np.random.randint(1,3)
            indici_usati_scelti = np.random.choice(asset_usati, size=numero_scelte, replace=False)

            numero_nuovi=2-indici_usati_scelti.size
            indici_nuovi_scelti=np.random.choice(asset_nuovi, size=numero_nuovi, replace=False)
            indici_totali=np.concatenate((indice1_array,indici_usati_scelti))
            indici_totali=np.concatenate((indici_totali, indici_nuovi_scelti))
            

            
            
        elif posti_liberi_portafoglio==0:
            numero_scelte=np.random.randint(0,2)
            if numero_scelte>0:
                indice_nuovo_scelto=np.random.choice(asset_nuovi,size=1)
                indice_usato_scelto=np.random.choice(asset_usati, size=1)
                indici_totali=np.concatenate((indice1_array,indice_usato_scelto))
                indici_totali=np.concatenate((indici_totali, indice_nuovo_scelto))
            elif numero_scelte==0:
                indici_usati_scelti=np.random.choice(asset_usati, size=2, replace=False)
                indici_totali=np.concatenate((indice1_array,indici_usati_scelti))
                





            
            
            
            
        
    






























    
    






Questo è il rendimento annuo con il portafoglio inizale:0.14288422773864512
Questo è la funzione inziale, rispetto al portafolgio inizale:0.06464883678399176


In [136]:

print(x.size)

2


In [137]:
print(np.random.randint(1,4))

3


In [150]:
indici_usati_scelti = np.array([1, 4])
indici_nuovi_scelti = np.array([20])

indici_totali = np.concatenate([indici_usati_scelti, indici_nuovi_scelti])
